# Knee v6: cache with K=2 (two series per plane), CPU only
Run with Accelerator = None (no GPU is needed, so it should not touch the GPU quota). Internet can be off.
Same cache as before except `--K 2`: each study becomes [6, 24, 160, 160] uint8 = Sagittal x2, Coronal x2, Axial x2 (slots ordered plane-major: 0-1 sagittal, 2-3 coronal, 4-5 axial).
Disk: 4407 studies x 6 x 24 x 160 x 160 = ~16.2 GB (the /kaggle/working limit is ~20 GB, so nothing else may be written there).
Cell 1 first shows how many studies even HAVE a second series per plane: if that is low, K=2 adds little and you can stop before caching.
Cell 2 times a 24-study smoke run (separating the one-off start-up from the per-study time) and projects total time and disk; the full run (cell 3) only starts if both fit.
Output folder: `cache/` (with `cache_index.csv`). Commit with Save & Run All; the committed output is what the training notebooks attach.

In [1]:
import os, sys, re, time, subprocess
from pathlib import Path
import numpy as np, pandas as pd
W = Path(os.environ.get("KNEE_WORK", "/kaggle/working"))
CODE = Path(os.environ.get("KNEE_CODE", "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"))
DATA = Path(os.environ.get("KNEE_DATA", "/kaggle/input/competitions/rsna-knee-abnormality-detection"))
for n, p in dict(CODE=CODE, DATA=DATA).items(): assert p.exists(), f"{n} not found: {p}"
assert (CODE / "cache_data.py").exists()
OUT = W / "cache"; D, SIZE, K = 24, 160, 2
ts = pd.read_csv(DATA / "train_series.csv")
n_studies = ts.StudyInstanceUID.nunique()
print("studies:", n_studies, "| cpus:", os.cpu_count())
# how many studies have a 2nd series per plane (the ceiling of what K=2 can add)
g = ts.groupby(["StudyInstanceUID", "Anatomical_Plane"]).size().unstack(fill_value=0)
for p in g.columns:
    print(f"{p:10s} studies with >=1 series: {(g[p] >= 1).mean():5.1%} | with >=2 series: {(g[p] >= 2).mean():5.1%}")
print(f"studies with a 2nd series in at least one plane: {(g >= 2).any(axis=1).mean():.1%}")
print("If the >=2 numbers are small (say under ~30%), K=2 can only help a minority of studies: consider stopping here.")
print(f"disk estimate: {n_studies * 3 * K * D * SIZE * SIZE / 1e9:.1f} GB")

studies: 4407 | cpus: 4
Axial      studies with >=1 series: 100.0% | with >=2 series: 24.8%
Coronal    studies with >=1 series: 100.0% | with >=2 series: 74.5%
Sagittal   studies with >=1 series: 100.0% | with >=2 series: 97.3%
studies with a 2nd series in at least one plane: 100.0%
If the >=2 numbers are small (say under ~30%), K=2 can only help a minority of studies: consider stopping here.
disk estimate: 16.2 GB


In [2]:
# smoke run on 24 studies (outside /kaggle/working so it does not count toward the output size)
SMOKE = Path("/tmp/cache_smoke"); 
import shutil; shutil.rmtree(SMOKE, ignore_errors=True)
cmd = [sys.executable, str(CODE / "cache_data.py"), "--split", "train", "--data_dir", str(DATA), "--out", str(SMOKE),
       "--D", str(D), "--size", str(SIZE), "--K", str(K), "--workers", str(os.cpu_count() or 2), "--limit", "24"]
t0 = time.time(); r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(CODE)); t24 = time.time() - t0
print(r.stdout[-400:]); assert r.returncode == 0, r.stderr[-1500:]
mm = re.search(r"(\d+)/(\d+) studies \| (\d+)s", r.stdout)
pool_s = float(mm.group(3)) if mm else t24                      # time of the caching pool itself (cache_data.py prints it)
startup_s = max(t24 - pool_s, 0.0)                              # fixed cost: listing every series folder, paid once per run
proj_h = (startup_s + pool_s / 24 * n_studies) / 3600
disk = n_studies * 3 * K * D * SIZE * SIZE / 1e9
print(f"24 studies: pool {pool_s:.0f}s ({pool_s/24:.1f} s/study) + fixed start-up {startup_s:.0f}s -> projected full run {proj_h:.1f} h, disk {disk:.1f} GB")
a = np.load(next(SMOKE.glob("*.npy"))); print("array shape:", a.shape)
assert a.shape == (3 * K, D, SIZE, SIZE)
assert disk < 18.5, "cache would not fit in /kaggle/working"
assert proj_h < 10, "projected run is too close to the 12 h session limit: use --D 16 or a smaller --size"

24/24 studies | 25s
failed/empty studies: 0 | saved /tmp/cache_smoke/cache_index.csv

24 studies: pool 25s (1.0 s/study) + fixed start-up 265s -> projected full run 1.3 h, disk 16.2 GB
array shape: (6, 24, 160, 160)


In [3]:
cmd = [sys.executable, str(CODE / "cache_data.py"), "--split", "train", "--data_dir", str(DATA), "--out", str(OUT),
       "--D", str(D), "--size", str(SIZE), "--K", str(K), "--workers", str(os.cpu_count() or 2)]
t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(CODE))
for line in p.stdout:
    line = line.rstrip()
    if "/" in line and "|" in line and line.split("/")[0].isdigit():
        if int(line.split("/")[0]) % 250 == 0: print(line, flush=True)
    else: print(line, flush=True)
assert p.wait() == 0, "cache_data.py failed"
print(f"done in {(time.time()-t0)/3600:.2f} h")

250/4407 studies | 244s
500/4407 studies | 497s
750/4407 studies | 751s
1000/4407 studies | 986s
1250/4407 studies | 1228s
1500/4407 studies | 1498s
1750/4407 studies | 1725s
2000/4407 studies | 2020s
2250/4407 studies | 2280s
2500/4407 studies | 2528s
2750/4407 studies | 2796s
3000/4407 studies | 3055s
3250/4407 studies | 3324s
3500/4407 studies | 3569s
3750/4407 studies | 3808s
4000/4407 studies | 4084s
4250/4407 studies | 4313s
failed/empty studies: 0 | saved /kaggle/working/cache/cache_index.csv
done in 1.25 h


In [4]:
idx = pd.read_csv(OUT / "cache_index.csv")
n_npy = len(list(OUT.glob("*.npy")))
print("studies indexed:", len(idx), "| npy files:", n_npy, "| failed/empty:", int((idx.ok == 0).sum()))
assert n_npy == n_studies and len(idx) == n_studies
names = ["Sag-1", "Sag-2", "Cor-1", "Cor-2", "Ax-1", "Ax-2"]
print("share of studies with each slot filled:", {n: round(float(idx[f"present_{i}"].mean()), 3) for i, n in enumerate(names)})
a = np.load(next(OUT.glob("*.npy")), mmap_mode="r"); print("array shape:", a.shape); assert a.shape == (6, D, SIZE, SIZE)
tot = sum(f.stat().st_size for f in OUT.glob("*.npy")) / 1e9
print(f"cache size on disk: {tot:.1f} GB")
assert tot < 19, "output too close to the 20 GB limit"

studies indexed: 4407 | npy files: 4407 | failed/empty: 0
share of studies with each slot filled: {'Sag-1': 1.0, 'Sag-2': 0.973, 'Cor-1': 1.0, 'Cor-2': 0.745, 'Ax-1': 1.0, 'Ax-2': 0.248}
array shape: (6, 24, 160, 160)
cache size on disk: 16.2 GB
